# ch01 - RAG Setup (code review)
This notebook executes the snippets from `ch01_RAG_setup.asciidoc` to ensure the walkthrough runs end-to-end.
When no `OPENAI_API_KEY` is available, it falls back to a lightweight mocked client so the pipeline can be verified offline.


In [ ]:
import subprocess, sys

packages = [
    'numpy<2',
    'openai==1.12.0',
    'chromadb==0.4.22',
    'tiktoken==0.5.2',
    'python-dotenv==1.0.0',
    'httpx==0.27.0',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)


In [ ]:
import os, sys, subprocess
from pathlib import Path
from dotenv import load_dotenv

IN_COLAB = 'google.colab' in sys.modules
load_dotenv()

def locate_datasets():
    candidates = [
        Path('/content/datasets'),
        Path.cwd() / 'datasets',
        Path.cwd().parent / 'datasets',
    ]

    for path in candidates:
        target = path / 'text_files' / 'harry_potter_knowledge_base.txt'
        if target.exists():
            return path
    return None

DATA_ROOT = locate_datasets()

if DATA_ROOT is None and IN_COLAB:
    subprocess.run(
        [
            'bash',
            '-lc',
            'git clone --depth 1 --filter=blob:none --sparse https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git repo_tmp && cd repo_tmp && git sparse-checkout set datasets && cp -r datasets /content/datasets',
        ],
        check=True,
    )
    DATA_ROOT = locate_datasets()

if DATA_ROOT is None:
    raise FileNotFoundError('harry_potter_knowledge_base.txt not found in datasets directory')

DATA_ROOT


In [ ]:
import hashlib
from types import SimpleNamespace
import httpx
from openai import OpenAI

USE_REAL_CLIENT = bool(os.getenv('OPENAI_API_KEY'))

class FakeEmbeddings:
    def create(self, model, input):
        texts = [input] if isinstance(input, str) else list(input)
        data = []
        for text in texts:
            digest = hashlib.sha256(text.encode()).digest()
            vec = [int.from_bytes(digest[i:i+4], 'big') / 1e9 for i in range(0, 12, 4)]
            data.append(SimpleNamespace(embedding=vec))
        return SimpleNamespace(data=data)

class FakeChatCompletions:
    def create(self, model, messages):
        user_msg = ''
        for msg in messages:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                user_msg = msg.get('content', '')
        answer = '[mocked {model}] ' + (user_msg.split('Question:')[-1].strip() or 'No question provided')
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=answer))])

class FakeChat:
    def __init__(self):
        self.completions = FakeChatCompletions()

class FakeOpenAI:
    def __init__(self):
        self.embeddings = FakeEmbeddings()
        self.chat = FakeChat()

def build_client():
    if USE_REAL_CLIENT:
        return OpenAI(http_client=httpx.Client())
    return FakeOpenAI()

client = build_client()
embedding_model = 'text-embedding-3-small'


In [ ]:
from pathlib import Path

def chunk_text(text, size=1000, overlap=200):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            bp = text.rfind("\n\n", start, end)
            if bp == -1:
                bp = text.rfind('. ', start, end)
            if bp > start:
                end = bp + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end
    return chunks

file_path = DATA_ROOT / 'text_files' / 'harry_potter_knowledge_base.txt'
text = file_path.read_text(encoding='utf-8')
chunks = chunk_text(text)
len(chunks)


In [ ]:
import chromadb

def embed_and_store(chunks, db_path, collection_name):
    chroma = chromadb.PersistentClient(path=str(db_path))
    collection = chroma.get_or_create_collection(
        name=collection_name, metadata={'description': 'Harry Potter knowledge base'}
    )

    for i in range(0, len(chunks), 100):
        batch = chunks[i : i + 100]
        res = client.embeddings.create(model=embedding_model, input=batch)
        collection.add(
            ids=[f'chunk_{i+j}' for j in range(len(batch))],
            documents=batch,
            embeddings=[x.embedding for x in res.data],
            metadatas=[{'chunk_index': i + j} for j in range(len(batch))],
        )
    return collection

chroma_db_dir = Path('ch01_RAG_setup') / 'chroma_db'
collection = embed_and_store(chunks, chroma_db_dir, 'harry_potter_kb')
collection.count()


In [ ]:
def retrieve(question, top_k=3):
    q_emb = client.embeddings.create(model=embedding_model, input=question).data[0].embedding
    res = collection.query(query_embeddings=[q_emb], n_results=top_k, include=['documents'])
    return res['documents'][0]

question = 'Why did Uncle Vernon take the family to a hut in the middle of the sea?'
docs = retrieve(question)
docs


In [ ]:
def answer(question, docs):
    context = '\n\n---\n\n'.join(docs)
    prompt = f"""Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"""

    res = client.chat.completions.create(
        model='gpt-5-mini',
        messages=[{'role': 'user', 'content': prompt}],
    )
    return res.choices[0].message.content

answer_text = answer(question, docs)
answer_text


In [ ]:
assert docs, 'No documents returned from retrieval'
assert isinstance(answer_text, str) and answer_text.strip(), 'No answer text produced'
print('Ran with real OpenAI client' if USE_REAL_CLIENT else 'Ran with mocked OpenAI client')
print('Smoke test complete.')
